# start 

In [15]:
# -----######-----###### RENAME FINAL: FULL CONTROL + PUkw CORRECTED -----######-----######
import os
import pandas as pd
from tqdm import tqdm
from datetime import datetime

def _rename_2304_kwtagging_GET_renamed_files_from_txt(
    txt_path,
    custom_artist="DJ_Selphi",
    custom_genre="Salsa",
    custom_label="Bachata",
    custom_release_date="",
    custom_purchase_date=""
):
    """
    Rename files using a kw-tag structure with optional custom overrides.
    Pulls from TXT if any override is left blank.

    Args:
        txt_path (str): Path to UTF-16 tab-separated metadata file
        custom_artist (str): Optional override for artist (max 25 chars)
        custom_genre (str): Optional override for genre
        custom_label (str): Optional override for label
        custom_release_date (str): Optional override in YYYY_MM_DD format
        custom_purchase_date (str): Optional override in YYYY_MM_DD format

    Returns:
        pd.DataFrame: Original DataFrame + 'Renamed_Path' column
    """
    tqdm.pandas()

    df = pd.read_csv(
        txt_path,
        sep="\t",
        encoding='utf-16',
        engine='python',
        on_bad_lines='skip'
    )
    
    df.columns = df.columns.str.strip()

    def clean(s):
        return (
            str(s)
            .replace(" ", "_").replace("/", "___").replace(",", "_")
            .replace("(", "").replace(")", "").replace("!", "")
            .replace("&", "and").replace("’", "").replace("'", "")
            .replace("¿", "").replace("¡", "").replace(":", "")
            .replace(";", "").strip()
        )

    def extract_mix(title):
        title_lower = title.lower()
        if "remix" in title_lower or "mix" in title_lower:
            return clean(title)
        return "original"

    def format_filename(row):
        title = clean(row.get('Track Title', ''))[:25]
        remix = extract_mix(row.get('Track Title', ''))

        artist_val = clean(custom_artist)[:25] if custom_artist else clean(row.get('Artist', ''))[:25]
        genre_val = clean(custom_genre) if custom_genre else clean(row.get('Genre', ''))
        label_val = clean(custom_label) if custom_label else clean(row.get('Label', ''))

        # ✅ FIXED: Use Release Date column if custom_release_date not passed
        release_date_val = (
            custom_release_date if custom_release_date
            else pd.to_datetime(row.get('Release Date', ''), errors='coerce').strftime('%Y_%m_%d')
            if pd.notna(row.get('Release Date', '')) else 'NA'
        )

        key = clean(row.get('Key', 'NA')) if pd.notna(row.get('Key', '')) else 'NA'
        bpm = str(int(round(float(row.get('BPM', 0))))) if pd.notna(row.get('BPM', 0)) else 'NA'

        date_added = pd.to_datetime(row.get('Date Added', datetime.today()), errors='coerce').strftime('%Y_%m_%d')
        purchase_date_val = (
            custom_purchase_date if custom_purchase_date
            else pd.to_datetime(row.get('Purchased', datetime.today()), errors='coerce').strftime('%Y_%m_%d')
        )

        ext = os.path.splitext(row.get('Location', ''))[1]

        new_name = (
            f"TRkw_{title}_ARkw_{artist_val}_MXkw_{remix}_KYkw_{key}_"
            f"BPkw_{bpm}_GNkw_{genre_val}_LBkw_{label_val}_RYkw_{release_date_val}_"
            f"PYkw_{purchase_date_val}{ext}"
        )

        if len(new_name) > 240:
            new_name = new_name[:230] + ext

        return new_name

    new_paths = []
    log_long_names = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Renaming files"):
        original_path = str(row.get('Location', '')).strip()

        if not os.path.isfile(original_path):
            print(f"❌ File not found (Row {i}): {original_path}")
            new_paths.append(None)
            continue

        new_filename = format_filename(row)
        new_path = os.path.join(os.path.dirname(original_path), new_filename)

        try:
            os.rename(original_path, new_path)
            new_paths.append(new_path)
        except Exception as e:
            print(f"❌ Error renaming (Row {i}): {e}")
            new_paths.append(None)
            log_long_names.append({
                "Index": i,
                "OriginalPath": original_path,
                "IntendedFilename": new_filename,
                "Error": str(e)
            })

    df['Renamed_Path'] = new_paths

    if log_long_names:
        pd.DataFrame(log_long_names).to_csv("long_name_errors_log.csv", index=False)
        print("📁 Saved log of failed renames to: long_name_errors_log.csv")

    return df


In [17]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

txt_path ='/Users/yerik/Downloads/bp/band camp_third/third.txt'

df = _rename_2304_kwtagging_GET_renamed_files_from_txt(
    txt_path,
    custom_artist="",           # ✅ or "" for TXT
    custom_genre="",                # ✅ or "" for TXT
    custom_label="",      # ✅ or "" for TXT
    custom_release_date="",    # ✅ or "" for fallback
    custom_purchase_date="2025_05_20"              # ✅ "" uses 'Purchased' column
)


Renaming files: 100%|██████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 424.68it/s]


# check

In [12]:
import pandas as pd

txt_path = "/Users/yerik/Downloads/bp/v.txt"

# 🚀 FAST & FORGIVING READ
df = pd.read_csv(txt_path, sep="\t", encoding='latin1', engine='python', on_bad_lines='skip')
df.columns = df.columns.str.strip()  # 🧼 Clean whitespaces

# 🎯 SHOW COLUMNS
print("\n🔎 COLUMN NAMES:")
for i, col in enumerate(df.columns):
    print(f"{i:>2}: '{col}'")

df.head(3)



🔎 COLUMN NAMES:
 0: 'ÿþ# '
 1: ' T r a c k   T i t l e '
 2: ' A r t i s t '
 3: ' G e n r e '
 4: ' L a b e l '
 5: ' R e l e a s e   D a t e '
 6: ' A l b u m '
 7: ' R e m i x e r '
 8: ' F i l e   T y p e '
 9: ' B P M '
10: ' K e y '
11: ' R a t i n g '
12: ' A r t w o r k '
13: ' C o m m e n t s '
14: ' B i t r a t e '
15: ' Y e a r '
16: ' L o c a t i o n '
17: ' F i l e   N a m e '
18: ' B i t d e p t h '
19: ' T i m e '
20: ' D a t e   A d d e d '


,ÿþ# , T r a c k   T i t l e , A r t i s t , G e n r e , L a b e l , R e l e a s e   D a t e , A l b u m , R e m i x e r , F i l e   T y p e , B P M ,..., R a t i n g , A r t w o r k , C o m m e n t s , B i t r a t e , Y e a r , L o c a t i o n , F i l e   N a m e , B i t d e p t h , T i m e , D a t e   A d d e d 
0, 1 , B u m p , B r y n j o l f u r , H o u s e , P r o m   N i g h t   R e c o r d s , 2 0 2 3 - 0 5 - 1 2 , B r y n j o l f u r   -   S i n t é t i c o  ...," P r i n s   T h o m a s ,   C h i n a s k i ", A I F F , 1 2 5 . 0 0 ,...,           , , V i s i t   h t t p s : / / p r o m n i g h t..., 2 1 1 6   k b p s , 0 , / U s e r s / y e r i k / D o w n l o a d s /..., B r y n j o l f u r   -   B r y n j o l f u r..., 2 4 , 0 8 : 0 8 , 2 0 2 5 - 0 5 - 2 1 
1, 2 , P a i s a j e   S i n t é t i c o   F e a t ....," B r y n j o l f u r ,   F i n a l   G r r l ", H o u s e , P r o m   N i g h t   R e c o r d s , 2 0 2 3 - 0 5 - 1 2 , B r y n j o l f u r   -   S i n t é t i c o  ...," P r i n s   T h o m a s ,   C h i n a s k i ", A I F F , 1 2 3 . 0 0 ,...,           , , V i s i t   h t t p s : / / p r o m n i g h t..., 2 1 1 6   k b p s , 0 , / U s e r s / y e r i k / D o w n l o a d s /...," B r y n j o l f u r ,   F i n a l   G r r l  ...", 2 4 , 0 6 : 3 5 , 2 0 2 5 - 0 5 - 2 1 
2, 3 , P a i s a j e   S i n t é t i c o   ( C h i n...," B r y n j o l f u r ,   F i n a l   G r r l ,...", H o u s e , P r o m   N i g h t   R e c o r d s , 2 0 2 3 - 0 5 - 1 2 , B r y n j o l f u r   -   S i n t é t i c o  ...," P r i n s   T h o m a s ,   C h i n a s k i ", A I F F , 1 2 4 . 0 0 ,...,           , , V i s i t   h t t p s : / / p r o m n i g h t..., 2 1 1 6   k b p s , 0 , / U s e r s / y e r i k / D o w n l o a d s /...," B r y n j o l f u r ,   F i n a l   G r r l ,...", 2 4 , 0 5 : 2 1 , 2 0 2 5 - 0 5 - 2 1 
